In [ ]:
"""
sandbox_bweight_decoder.ipynb

Characterize encoding profiles of neurons most informative in decoding strategy.

Author: Stellina X. Ao
Created: 2026-07-29
Last Modified: 2026-07-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = (
    "20251027_152036"  # "20251024_142407"  # "20251028_140930" # "20251027_152036"
)

## init

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

fig, ax = plt.subplots(figsize=(4, 1.5), tight_layout=True)
ax.plot(encoder.trial_data["strategy"], linewidth=0.5, color="k")
ax.set_xlabel("trials")
ax.set_yticks([-1, 1], ["mf", "mb"])
ax.spines[["top", "left", "right"]].set_visible(False)


fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight" / "decoder_sort"
save_fig(fig, fpath, "strategy_trace.png")

In [ ]:
from core.data import get_strategy_filter_idxs

idxs_all = get_strategy_filter_idxs(encoder.trial_data, cond_balance=True)
idxs = np.sort(np.concatenate((idxs_all["mb"], idxs_all["mf"])))

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import KFold

X = encoder.robs[idxs]
y = encoder.trial_data["strategy"].iloc[idxs]

decoder = LogisticRegressionCV(Cs=np.logspace(-5, 5, 11, base=10)).fit(X, y)
scores_cv = np.zeros(5)

for i, (train_idx, test_idx) in enumerate(
    KFold(n_splits=5, shuffle=True, random_state=2).split(X, y)
):
    decoder = LogisticRegressionCV(Cs=np.logspace(-5, 5, 11, base=10)).fit(
        X[train_idx], y.iloc[train_idx]
    )
    scores_cv[i] = decoder.score(X[test_idx], y.iloc[test_idx])

print(scores_cv.mean())

In [ ]:
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR


# neurons with the largest weight, what is their task variable encoding like?
def plot_bweight_dsort(encoders):
    fig, axes = plt.subplots(ncols=3, figsize=(8, 3), tight_layout=True)

    for i, ((model, e), ax) in enumerate(zip(encoders.items(), axes.flat)):
        im = ax.imshow(
            e.encoder_weights[np.argsort(decoder.coef_.ravel()), 5:],
            aspect="auto",
            vmin=-1,
            vmax=1,
            cmap="coolwarm",
        )
        ax.set_xticks(
            ticks=range(encoder.num_tv),
            labels=encoder.dm_names[5:],
            fontsize=4,
            rotation=90,
        )
        ax.set_xlabel("task variables")
        ax.set_ylabel("neurons")
        ax.set_title(f"{model}", fontsize=5)

    fig.suptitle(
        f"encoding weight (sorted by decoding bweight, cv-acc {scores_cv.mean():.3f})",
        fontsize=5,
    )
    fig.colorbar(im)

    fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight" / "decoder_sort"
    save_fig(fig, fpath, "bweights_decoder_sort.png")


plot_bweight_dsort({"full": encoder, "mb": encoder_mb, "mf": encoder_mf})